In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Union, Sequence, Tuple
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import pandas as pd
import h5py
# ----------------------------------------------------------------------
# I/O helpers
# ----------------------------------------------------------------------

def load_coordinates(crd_file: Union[str, Path]) -> np.ndarray:
    """Return z‑coordinates, shape (nz,)."""
    crd_file = Path(crd_file)
    return np.loadtxt(crd_file, comments="#")

def load_datasets(
    dat_file: Union[str, Path], nz: int
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Read a single .dat file; return t, W, F.

    t : (nt,)                 – times
    W : (nt, nz)              – w(z,t)
    F : (nt, nz)              – f(z,t)
    """
    dat_file = Path(dat_file)
    times: list[float] = []
    w_chunks: list[np.ndarray] = []
    f_chunks: list[np.ndarray] = []

    with dat_file.open() as fp:
        for line in fp:
            if line.startswith("# dt="):
                *_, time_token = line.strip().split()
                times.append(float(time_token.split("=")[1]))

                next(fp)  # skip '# [ w, f ]'
                w_arr = np.empty(nz, dtype=float)
                f_arr = np.empty(nz, dtype=float)
                for i in range(nz):
                    w_val, f_val = map(float, next(fp).split())
                    w_arr[i] = w_val
                    f_arr[i] = f_val
                w_chunks.append(w_arr)
                f_chunks.append(f_arr)

    t = np.asarray(times)
    W = np.vstack(w_chunks)
    F = np.vstack(f_chunks)
    return t, W, F

def load_and_filter(part_dirs: Sequence[Union[str, Path]],
                    crd_name: str = "burgers_1D.crd",
                    dat_name: str = "burgers_1D.dat",
                    window: float = 5.0,
                    t_end: float | None = None,
                    mode: str = "last") -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    mode: "all", "first", "last"
    """
    part_dirs = [Path(p) for p in part_dirs]
    if not part_dirs:
        raise ValueError("No directories provided")

    z = load_coordinates(part_dirs[0] / crd_name)
    nz = len(z)

    t_all: list[np.ndarray] = []
    W_all: list[np.ndarray] = []
    F_all: list[np.ndarray] = []

    for d in part_dirs:
        t_i, W_i, F_i = load_datasets(d / dat_name, nz)
        t_all.append(t_i)
        W_all.append(W_i)
        F_all.append(F_i)

    t_full = np.concatenate(t_all)
    W_full = np.concatenate(W_all, axis=0)
    F_full = np.concatenate(F_all, axis=0)

    indices = np.argsort(t_full)
    t_full = t_full[indices]
    W_full = W_full[indices]
    F_full = F_full[indices]

    if mode == "all":
        return z, t_full, W_full, F_full
    elif mode == "first":
        t_start = t_full[0]
        t_end = t_start + window
    elif mode == "last":
        t_end = t_full[-1] if t_end is None else t_end
        t_start = t_end - window
    else:
        raise ValueError("Invalid mode. Choose from: 'all', 'first', 'last'.")

    mask = (t_full >= t_start) & (t_full <= t_end)
    return z, t_full[mask], W_full[mask], F_full[mask]

# ----------------------------------------------------------------------
# Interpolation helpers
# ----------------------------------------------------------------------

def cell_centres(z: np.ndarray, n_cells: int) -> np.ndarray:
    """Return the centres of ``n_cells`` equally spaced control volumes."""
    z_min, z_max = z[0], z[-1]
    dz = (z_max - z_min) / n_cells
    return z_min + (0.5 + np.arange(n_cells)) * dz

def interpolate_field(
    z_src: np.ndarray,
    field_src: np.ndarray,
    z_dst: np.ndarray,
    kind: str,
) -> np.ndarray:
    """
    Interpolate (or average) a 2-D field in the *space* direction only.

    Parameters
    ----------
    z_src      : (nz,)        nodal coordinates of the DNS data
    field_src  : (nt, nz)     snapshots to be mapped
    z_dst      : (m,)         target coordinates (typically cell centres)
    kind       : str          one of {'linear', 'nearest', 'cubic', 'average'}

        'linear'   - first-order continuous interpolation (SciPy)
        'nearest'  - piecewise-constant nearest-neighbour (SciPy)
        'cubic'    - C¹-continuous cubic spline (SciPy, default)
        'average'  - conservative block-average *exactly* suited to FVM
                     • requires that (len(z_src)-1) % len(z_dst) == 0
                     • introduces **zero** spatial error provided the DNS
                       mesh is a refinement by an integer factor.
    """
    nt, _ = field_src.shape

    # ------------------------------------------------------------------
    # 1.  Conservative block-averaging (FVM cell averages)
    # ------------------------------------------------------------------
    if kind == "average":
        n_src_cells = len(z_src) - 1          # nodal → cell count
        n_dst_cells = len(z_dst)

        if n_src_cells % n_dst_cells:
            raise ValueError(
                "'average' kind requires the destination grid to be an "
                "integer coarsening of the source grid "
                f"({n_src_cells=} not divisible by {n_dst_cells=})."
            )

        ratio = n_src_cells // n_dst_cells    # fine-to-coarse ratio

        # fine cell averages: midpoint rule ½(u_i + u_{i+1})  (2nd order)
        fine_avg = 0.5 * (field_src[:, :-1] + field_src[:, 1:])  # (nt, n_src_cells)

        # block average over 'ratio' consecutive fine cells
        result = fine_avg.reshape(nt, n_dst_cells, ratio).mean(axis=2)
        return result
    else:
        # ------------------------------------------------------------------
        # 2.  Interpolation via SciPy for the other three cases
        # ------------------------------------------------------------------
        print(f"Interpolating {kind} from {len(z_src)} to {len(z_dst)} points.")
        result = np.empty((nt, len(z_dst)), dtype=field_src.dtype)
        for k in range(nt):
            interp = interp1d(
                z_src,
                field_src[k, :],
                kind=kind,
                fill_value="extrapolate",
                bounds_error=False,
            )
            result[k, :] = interp(z_dst)
        return result

def interpolate_in_time(
    t_src: np.ndarray,
    field_src: np.ndarray,
    dt_new: float,
    kind: str = "linear",
) -> tuple[np.ndarray, np.ndarray]:
    """
    Uniformly resample a 2-D field in the *time* direction.

    Parameters
    ----------
    t_src     : (nt,)         original time stamps (must be sorted ascending)
    field_src : (nt, m)       snapshots at each t_src
    dt_new    : float         desired new time step
    kind      : str           interpolation kind for SciPy interp1d
                             (e.g. 'linear', 'nearest', 'cubic')

    Returns
    -------
    t_dst     : (nt_new,)     new, evenly spaced time stamps
    field_dst : (nt_new, m)   field interpolated in time
    """
    # build new time vector
    t0, t1 = t_src[0], t_src[-1]
    # ensure inclusive of the last point
    nt_new = int(np.floor((t1 - t0) / dt_new)) + 1 + 1
    t_dst = t0 + np.arange(nt_new) * dt_new

    # interpolate along axis=0 (time axis)
    interp = interp1d(
        t_src,
        field_src,
        axis=0,
        kind=kind,
        bounds_error=False,
        fill_value="extrapolate",
    )
    field_dst = interp(t_dst)
    return t_dst, field_dst

def compute_normalized_J(
    z: np.ndarray,
    t: np.ndarray,
    W: np.ndarray,
    center: float = 1.0,
    sigma: float = 0.1,
    weight_scale: float = 50.0
) -> float:
    """
    Compute the normalized QoI J̄ from wall-normal velocity data.

    Parameters
    ----------
    z : ndarray of shape (nz,)
        Spatial coordinates (assumed uniform or sorted).
    t : ndarray of shape (nt,)
        Time vector.
    W : ndarray of shape (nt, nz)
        Velocity field w(y,t), organized as time-major.
    center : float
        Center of the Gaussian weight function (typically 1.0).
    sigma : float
        Standard deviation of the Gaussian weight function (default: 0.1).
    weight_scale : float
        Exponential decay factor (default: 50, i.e., e^{-50y²}).

    Returns
    -------
    J : float
        Normalized time-averaged quantity of interest.
    """
    if W.shape != (len(t), len(z)):
        raise ValueError(f"Expected W shape ({len(t)}, {len(z)}), got {W.shape}")

    # Compute std in time for each spatial point
    w_std = np.std(W, axis=0)

    # Gaussian weighting function: e^{-weight_scale*(y - center)^2}
    weight = np.exp(-weight_scale * (z - center)**2)

    # Denominator: spatial integral of σ(y)^4 * weight(y)
    denom = np.trapz(w_std**4 * weight, x=z)

    # Numerator: double integral over time and space of w^4(y,t) * weight(y)
    integrand = W**4 * weight[None, :]  # shape (nt, nz)
    num = np.trapz(np.trapz(integrand, x=z, axis=1), x=t)

    # Time span
    T = t[-1] - t[0]

    # return num / (T * denom)
    return num / T, num / (T * denom)


def topHatFilter(z_src: np.ndarray, field_src: np.ndarray, z_dst: np.ndarray, threshold: float) -> np.ndarray:
    """
    Perform top-hat filtering of `field_src` from `z_src` onto coarse `z_dst`.

    Parameters
    ----------
    z_src      : (nz,)         Fine grid points (original DNS resolution).
    field_src  : (nt, nz)      Time-dependent field at fine resolution.
    z_dst      : (m,)          Coarse grid points (cell centers).
    threshold  : float         Filter width (typically 2 * delta_x of coarse grid).

    Returns
    -------
    field_flt : (nt, m)        Filtered field at coarse resolution.
    """
    nt, nz = field_src.shape
    m = len(z_dst)
    field_flt = np.zeros((nt, m))

    for j in range(m):
        # Get the center of the top-hat filter
        center = z_dst[j]
        # Find indices within the filter width
        mask = (z_src >= center - threshold/2) & (z_src <= center + threshold/2)
        # Apply uniform averaging in space
        if np.any(mask):
            field_flt[:, j] = np.mean(field_src[:, mask], axis=1)
        else:
            # Fallback to nearest if no points are found (should rarely happen)
            nearest_idx = np.argmin(np.abs(z_src - center))
            field_flt[:, j] = field_src[:, nearest_idx]
    return field_flt


In [ ]:
timeLength = 5
coarseNs = [1024]  # coarse mesh size
fineNs = [2 * n for n in coarseNs]
refN = 4096
scheme = "linear"  # 'linear', 'nearest', 'cubic', 'average'
dt_new = 1e-5

In [ ]:
dirs = ["Raw Data/"]
z, t, W, F = load_and_filter(dirs, mode="first", window=timeLength)

print("shape of t", np.shape(t))
print("shape of W", np.shape(W))

In [ ]:
t_resampled, W_resampled = interpolate_in_time(t, W, dt_new, kind="linear")
_, F_resampled           = interpolate_in_time(t, F, dt_new, kind="linear")

print("Original time steps:", len(t), "→ dt ≃", np.diff(t).mean())
print("Resampled time steps:", len(t_resampled), "→ dt =", dt_new)



In [ ]:
# t_resampled = t
# W_resampled = W
# F_resampled = F
# print("Original time steps:", len(t), "→ dt ≃", np.diff(t).mean())
# print("Resampled time steps:", len(t_resampled), "→ dt =", dt_new)


In [ ]:
# Inspired from DNS Data
zCellCenterRef = cell_centres(z, refN)
refU    = interpolate_field(z, W_resampled, zCellCenterRef, kind="linear")
refF    = interpolate_field(z, F_resampled, zCellCenterRef, kind="linear")

print(refF.shape)

In [ ]:
# saveDirectoryRef = "dnsData/refInspiredDNS/" + str(int(timeLength*10)) + "/dt" + str(dt_new) + "/" + scheme + "/f" + str(refN).zfill(4) + "/"
# Path(saveDirectoryRef).mkdir(parents=True, exist_ok=True)

# h5fileFull  = saveDirectoryRef + "dnsFullRawData.h5"
# h5fileShort = saveDirectoryRef + "dnsShortRawData.h5"

# # ------------------------------------------------------------------
# #  write – one dataset per array, **no compression**
# # ------------------------------------------------------------------
# with h5py.File(h5fileFull, "w") as h5:
#     h5.create_dataset("t", data=t_resampled)            # (nt,)
#     print("Part t written")
#     h5.create_dataset("z", data=zCellCenterRef)         # (nz,)
#     print("Part z written")
#     h5.create_dataset("U", data=refU)                   # (nt, nz)
#     print("Part U written")
#     h5.create_dataset("F", data=refF)                   # (nt, nz)
#     print("Part F written")

# with h5py.File(h5fileShort, "w") as h5:
#     h5.create_dataset("U0", data=refU[0,:])             # (nt, nz)
#     print("Part U0 written")
#     h5.create_dataset("F", data=refF)                   # (nt, nz)
#     print("Part F written")


In [ ]:
scheme = "topHat"
for coarseN, fineN in zip(coarseNs, fineNs):
    print(f"Coarse mesh size: {coarseN}, Fine mesh size: {fineN}")

    saveDirectoryCoarse = "dnsData/t" + str(int(timeLength*10)) + "/dt" + str(dt_new) + "/" + scheme + "/f" + str(fineN).zfill(4) + "c" + str(coarseN).zfill(4) + "/coarse/"
    saveDirectoryFine   = "dnsData/t" + str(int(timeLength*10)) + "/dt" + str(dt_new) + "/" + scheme + "/f" + str(fineN).zfill(4) + "c" + str(coarseN).zfill(4) + "/fine/"

    zCellCenterCoarse = cell_centres(z, coarseN)
    zCellCenterFine   = cell_centres(z, fineN)

    dxCoarse = zCellCenterCoarse[1] - zCellCenterCoarse[0]
    dxFine   = zCellCenterFine[1]   - zCellCenterFine[0]
    thresholdCoarse = 2 * dxCoarse
    thresholdFine   = 2 * dxFine

    coarseW = topHatFilter(zCellCenterRef, refU, zCellCenterCoarse, thresholdCoarse)
    coarseF = topHatFilter(zCellCenterRef, refF, zCellCenterCoarse, thresholdCoarse)
    fineW   = topHatFilter(zCellCenterRef, refU, zCellCenterFine,   thresholdFine)
    fineF   = topHatFilter(zCellCenterRef, refF, zCellCenterFine,   thresholdFine)


    # coarseW = interpolate_field(zCellCenterRef, refU, zCellCenterCoarse, kind=scheme)
    # coarseF = interpolate_field(zCellCenterRef, refF, zCellCenterCoarse, kind=scheme)
    # fineW   = interpolate_field(zCellCenterRef, refU, zCellCenterFine, kind=scheme)
    # fineF   = interpolate_field(zCellCenterRef, refF, zCellCenterFine, kind=scheme)



    Path(saveDirectoryCoarse).mkdir(parents=True, exist_ok=True)
    Path(saveDirectoryFine).mkdir(parents=True, exist_ok=True)

    with h5py.File(saveDirectoryCoarse + "dnsFullData.h5", "w") as h5:
        h5.create_dataset("t", data=t_resampled)                    # (nt,)
        h5.create_dataset("z", data=zCellCenterCoarse)              # (nz,)
        h5.create_dataset("U", data=coarseW)                        # (nt, nz)
        h5.create_dataset("F", data=coarseF)                        # (nt, nz)

    with h5py.File(saveDirectoryFine + "dnsFullData.h5", "w") as h5:
        h5.create_dataset("t", data=t_resampled)                    # (nt,)
        h5.create_dataset("z", data=zCellCenterFine)                # (nz,)
        h5.create_dataset("U", data=fineW)                          # (nt, nz)
        h5.create_dataset("F", data=fineF)                          # (nt, nz)

    with h5py.File(saveDirectoryCoarse + "dnsShortData.h5", "w") as h5:
        h5.create_dataset("U0", data=coarseW[0,:])                  # (nt, nz)
        h5.create_dataset("F", data=coarseF)                        # (nt, nz)

    with h5py.File(saveDirectoryFine + "dnsShortData.h5", "w") as h5:
        h5.create_dataset("U0", data=fineW[0,:])                    # (nt, nz)
        h5.create_dataset("F", data=fineF)                          # (nt, nz)

